# Adjoint probes and executable falsifiers

This lesson starts from energy, derives a two-query probe, averages sign probes, and finishes by breaking and restoring a kernel witness.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import IntSlider, interact

PAPER = Path.cwd()
sys.path.insert(0, str(PAPER / 'code'))
from lggr_vi_core import (
    ObservationSystem, break_and_recover_kernel_vector,
    central_directional_derivative, full_rademacher_directions,
    rademacher_adjoint_estimate,
)

system = ObservationSystem([[2.0, -1.0], [0.0, 0.5]])
state = np.array([0.8, -0.4])
truth = system.adjoint_signal(state)
truth

## Level 1 — energy answers directional questions

For $f(x)=\tfrac12\|Qx\|_2^2$, two symmetric evaluations give

$$\frac{f(x+cd)-f(x-cd)}{2c}=d^{\mathsf T}Q^{\mathsf T}Qx.$$

In [ ]:
direction = np.array([-1.0, 1.0])
measured = central_directional_derivative(system, state, direction, step=1e-5)
expected = direction @ truth
assert np.isclose(measured, expected)
{'two_query_measurement': measured, 'adjoint_pairing': expected}

## Level 2 — sign probes

Multiplying the directional measurement by a sign vector $\delta$ gives $\delta\delta^{\mathsf T}Q^{\mathsf T}Qx$. Averaging all signs cancels every off-diagonal product. Use the slider to see why an early prefix can be poor even though the full average is exact.

In [ ]:
directions = full_rademacher_directions(2)
def plot_probe_prefix(count):
    estimate = rademacher_adjoint_estimate(system, state, directions[:count])
    error = np.linalg.norm(estimate - truth)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(['first', 'second'], estimate, color='tab:orange', label='estimate')
    ax.scatter(['first', 'second'], truth, color='black', zorder=3, label='truth')
    ax.set(title=f'{count} sign probes; error={error:.3g}', ylabel='component')
    ax.legend(); display(fig); plt.close(fig)
    return estimate

@interact(count=IntSlider(min=1, max=len(directions), step=1, value=len(directions)))
def probe_prefix(count):
    return plot_probe_prefix(count)

### Static fallback

The complete sign family below uses the same plotting function and remains visible when interactive widgets are unavailable.

In [ ]:
static_estimate = plot_probe_prefix(len(directions))
assert np.allclose(static_estimate, truth)
static_estimate

## Level 3 — break the claim on purpose

A test that only passes on the intended matrix might be vacuous. We add one watcher coefficient, demand detection, restore the coefficient, and demand recovery.

In [ ]:
receipt = break_and_recover_kernel_vector(
    matrix=[[1.0, 0.0]],
    claimed_kernel_vector=[0.0, 1.0],
    entry=(0, 1),
    perturbation=0.25,
)
assert receipt.baseline_hidden and receipt.mutation_detected and receipt.restored_hidden
receipt

## Level 4 — proof boundary

$\ker(Q^{\mathsf T}Q)=\ker Q$ follows from $x^{\mathsf T}Q^{\mathsf T}Qx=\|Qx\|_2^2$. The Lean file checks consequences of this identity under an explicit hypothesis; the manuscript supplies the real-matrix proof.

## Level 5 — professor prompts

1. Replace full sign averaging by random sampling and derive a concentration bound.
2. Let $Q$ depend on earlier probe outcomes. Identify where the fixed-operator proof fails.
3. Add observation noise and separate bias from variance in the centered estimator.
4. Design an added watcher row that maximizes the least positive singular value under a norm budget.

These are research questions, not results of this notebook.